### Prerequisites
- An Azure Event Hubs namespace with a Kafka-compatible endpoint enabled
- A Shared Access Key with `Listen` permissions on the `orders` topic
- A Databricks cluster with DBR 10.4+ (Kafka connector is bundled — no extra libraries needed)

# Azure Event Hubs → Spark Ingestion Explorer

This notebook explores end-to-end ingestion of restaurant order events from **Azure Event Hubs** using its Kafka-compatible endpoint. It covers connection setup, raw message decoding, JSON schema parsing, and flattening into a clean DataFrame — serving as the basis for the pipeline's bronze layer.

## Steps
| # | Cell | Description |
|---|------|-------------|
| 1 | Configuration | Define Event Hubs namespace, topic, and Kafka consumer options (SASL/SSL via `PlainLoginModule`) |
| 2 | Raw Ingestion | Batch-read raw Kafka messages — `key` and `value` are binary |
| 3 | Decode | Cast binary `key`/`value` columns to strings |
| 4 | Parse JSON | Deserialize the `value_str` payload using a defined schema |
| 5 | Flatten | Select all parsed fields and rename `timestamp` → `order_timestamp` |
| 6 | Final Code | Consolidated single-cell version of the full ingestion logic |

## Order Event Schema
Each message in the `orders` topic carries the following fields:
- `order_id`, `restaurant_id`, `customer_id` — identifiers
- `order_timestamp`, `created_at` — timing
- `order_type`, `order_status`, `payment_method` — status fields
- `items` — array of `{item_id, name, category, quantity, unit_price, subtotal}`
- `total_amount` — order total

> **Reference:** [Azure Databricks – Event Hubs Integration](https://learn.microsoft.com/en-us/azure/databricks/ldp/event-hubs)

---
## Step 1 — Configuration
Define the Event Hubs namespace, topic name, and Kafka consumer options. The `KAFKA_OPTIONS` dict is shared across all read operations. Authentication uses SASL/SSL with the `PlainLoginModule` and a `$ConnectionString` username — the standard pattern for Event Hubs Kafka endpoints.

In [0]:
# Event Hubs configuration
EH_NAMESPACE = dbutils.secrets.get(scope='restaurantproject', key='EVENTHUB_NAMESPACE') 	
EH_NAME      = dbutils.secrets.get(scope='restaurantproject', key='EVENTHUB_NAME') 	
EH_CONN_STR  = dbutils.secrets.get(scope='restaurantproject', key='EVENTHUB_CONN_STR') 	

# Kafka Consumer configuration
KAFKA_OPTIONS = {
  "kafka.bootstrap.servers"  : f"{EH_NAMESPACE}.servicebus.windows.net:9093",
  "subscribe"                : EH_NAME,
  "kafka.sasl.mechanism"     : "PLAIN",
  "kafka.security.protocol"  : "SASL_SSL",
  "kafka.sasl.jaas.config"   : f"kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username=\"$ConnectionString\" password=\"{EH_CONN_STR}\";",
  "kafka.request.timeout.ms" : "60000",         # Maximum time to wait for a response from Kafka i.e. 60 seconds
  "kafka.session.timeout.ms" : "30000",         # Maximum time to wait for a group to rebalance i.e. 30 seconds
  "maxOffsetsPerTrigger"     : "50000",         # Maximum number of records to read from Kafka in each micro-batch
  "failOnDataLoss"           : "true",          # On data loss, throw an error
  "startingOffsets"          : "earliest"       # Latest offset to start reading from Kafka
}

---
## Step 2 — Raw Ingestion
Read a batch snapshot of all messages from the `orders` topic using `startingOffsets: earliest`. At this stage `key` and `value` are raw `BinaryType` — Kafka's wire format. The schema also includes Kafka metadata columns: `topic`, `partition`, `offset`, `timestamp`, and `timestampType`.

In [0]:
df_raw=spark.read.format("kafka").options(**KAFKA_OPTIONS).load()
display(df_raw.printSchema)
display(df_raw)

---
## Step 3 — Decode Binary Columns
Cast the binary `key` and `value` columns to UTF-8 strings. This makes the JSON payload in `value_str` human-readable and ready for schema-based parsing in the next step.

In [0]:
from pyspark.sql.functions import col

df_raw = (
    df_raw
    .withColumn("key_str", col("key").cast("string"))
    .withColumn("value_str", col("value").cast("string"))
)
display(df_raw)

---
## Step 4 — Parse JSON Payload
Apply an explicit Spark DDL schema to deserialize `value_str` into a structured `value_parsed` column via `from_json()`. Providing the schema upfront avoids a full data scan for inference and ensures correct types (e.g. `TIMESTAMP` vs `STRING` for date fields, `DOUBLE` for monetary values).

In [0]:
from pyspark.sql.functions import from_json

schema="""
order_id STRING, 
order_timestamp TIMESTAMP, 
restaurant_id STRING, 
customer_id STRING, 
order_type STRING, 
items ARRAY<STRUCT<item_id STRING, name STRING, category STRING, quantity INTEGER, unit_price DOUBLE, subtotal DOUBLE>>, 
total_amount DOUBLE, 
payment_method STRING, 
order_status STRING, 
last_updated TIMESTAMP
"""

df_raw = df_raw.withColumn("value_parsed", from_json(col("value_str"), schema))
display(df_raw)


---
## Step 5 — Flatten Parsed Fields
Expand the nested `value_parsed` struct into top-level columns with `select("value_parsed.*")`. The `timestamp` field is renamed to `order_timestamp` to avoid shadowing Kafka's built-in `timestamp` metadata column.

In [0]:
df_parsed=( df_raw.select("value_parsed.*")
                  .withColumnRenamed("timestamp","order_timestamp"))
display(df_parsed)

---
## Step 6 — Final Consolidated Code
Single-cell version combining all previous steps into one clean pipeline. This is the canonical reference for implementing the bronze streaming table in the Lakeflow pipeline — copy this logic into the pipeline source file when ready.

In [0]:
## Final Code
from pyspark.sql.functions import col,from_json

df_raw=spark.read.format("kafka").options(**KAFKA_OPTIONS).load()

schema="""
order_id STRING, 
order_timestamp TIMESTAMP, 
restaurant_id STRING, 
customer_id STRING, 
order_type STRING, 
items ARRAY<STRUCT<item_id STRING, name STRING, category STRING, quantity INTEGER, unit_price DOUBLE, subtotal DOUBLE>>, 
total_amount DOUBLE, 
payment_method STRING, 
order_status STRING, 
last_updated TIMESTAMP
"""

df_parsed = (
    df_raw
    .withColumn("value_str", col("value").cast("string"))
    .withColumn("value_parsed", from_json(col("value_str"), schema))
    .select(col("value_parsed.*"))
    .withColumnRenamed("timestamp","order_timestamp")
)
display(df_parsed)